# ML Project / NASDAQ Prediction and Analysis

**Objective** : Predicting if the Nasdaq-100 (`^NDX`) goes up or down **during the New York session of day `t`** (from the open at 09:30 ET to the close at 16:00 ET), using only information known **before 09:00 ET of day `t`**. The main new information is how the NQ futures traded overnight (Asia and Europe); the daily data (stocks, macro) is used as context from the day before.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import requests 
import io
import os
import kagglehub
from kagglehub import KaggleDatasetAdapter 
from sklearn.model_selection import TimeSeriesSplit

In [ ]:
path_to_data = ("./data")

## Data

We take the current Nasdaq-100 list from Wikipedia. This means we use today's companies for the whole period, so there's some survivorship bias (we'll mention it in the report).

In [ ]:
html = requests.get("https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies", headers={"User-Agent": "Mozilla/5.0"}).text
table = pd.read_html(io.StringIO(html))
ndx = next(t for t in table if "Ticker" in t.columns)
TICKERS = ndx["Ticker"].str.replace(".", "-", regex=False).tolist()

In [ ]:

NQ = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "tgtanalytics/nq-futures-1min-bar-2022-2025",
  "Dataset_NQ_1min_2022_2025.csv"
)

In [ ]:
TICKERS

On top of the stocks we add some macro data: volatility indices (VIX, VXN), treasury yields, oil, gold, silver, the dollar and Bitcoin. The idea is to give the model some context about the market, not just prices.

In [ ]:
NASDAQ_FEATURES = {
    # --- INDICES ET VOLATILITÉ ---                    
    "^IXIC": "Nasdaq_composite",              # Broad tech market index
    "^VXN": "Nasdaq_volatility",              # Nasdaq-100 implied volatility (Tech fear index)
    "^VIX": "Volatility",                     # S&P 500 implied volatility (Market fear index)

    # --- interest ---
    "^IRX": "3month_treasury_yield",          # 3-month Treasury yield
    "^FVX": "5y_treasury_yield",              # 5-year Treasury yield
    "^TNX": "10y_treasury_yield",             # 10-year Treasury yield (Tech discounting rate)
    "^TYX": "30y_treasury_yield",             # 30-year Treasury yield

    # --- MATIÈRES PREMIÈRES ET DEVISES ---
    "DX-Y.NYB": "US_dollar",                  # US dollar index (DXY)
    "CL=F": "crude_oil",                      # WTI crude oil futures
    "GC=F": "gold",                           # Gold futures
    "SI=F": "silver",                         # Silver futures
    "BTC-USD": "Bitcoin",                     # Bitcoin USD (Risk-on sentiment proxy)
}
tickers1 = list(NASDAQ_FEATURES.keys())

In [ ]:
TICKERS = list(dict.fromkeys(TICKERS + tickers1))

In [ ]:
df_nasdaq = yf.download("^NDX", start="2022-10-01", end="2025-12-12",
                     auto_adjust=True, progress=False)[['Open', 'Close','Volume']]
print(df_nasdaq.columns)
print(df_nasdaq.tail())

In [ ]:
df_nasdaq.head(20)

In [ ]:
raw = yf.download(TICKERS, start="2022-10-01", end="2025-12-12")[['Open', 'Close','Volume']]

In [ ]:
raw.head(10)

Bitcoin trades 24/7, so before the join we keep its close on its own calendar (weekends included). We use it later for the Bitcoin features.

In [ ]:
# BTC close on its own daily calendar, weekends included
btc_close = raw[("Close", "BTC-USD")].dropna()

We join everything on the Nasdaq-100 dates (inner join). Bitcoin trades on weekends but the index doesn't, so this way we only keep real trading days and we don't need to fill anything.

In [ ]:
raw = pd.concat((df_nasdaq,raw),axis=1,join='inner',)

In [ ]:
raw.loc[:, (slice(None), 'BTC-USD')]

In [ ]:
raw

### Missing values

Some companies are very recent (ARM, CRWV, SNDK...), so they have almost no history. We drop the columns with less than 40% of data. We picked this threshold by looking at which columns were left.

We also drop GOOG (almost the same as GOOGL) and ^IXIC (it moves with the Nasdaq-100), they don't add anything new.

In [ ]:
dropping = raw.isnull().sum().sort_values(ascending=False)/len(raw)

In [ ]:
dropping

In [ ]:
raw_drop = raw.dropna(axis=1, thresh=int(0.4 * len(raw)))
# GOOG is basically GOOGL and ^IXIC moves with ^NDX, so they add nothing
raw_drop = raw_drop.drop(columns=["GOOG", "^IXIC"], level=1)

In [ ]:
print(list(set(raw.columns.tolist()) - set(raw_drop.columns.tolist())))

In [ ]:
try:
    raw_drop.loc[:, (slice(None), "SPCX")]
except KeyError:
    print("SPCX not in raw_drop: it was dropped")

In [ ]:
raw.loc[:, (slice(None), "SPCX")]

In [ ]:
raw_drop.describe()

In [ ]:
from fastai.tabular.all import add_datepart

raw_drop = raw_drop.reset_index()
raw_time = add_datepart(raw_drop, 'Date', drop=False)

From the date we keep only `Dayofweek`. The other date columns (year, month, elapsed...) are trends or noise for this target.

In [ ]:
date_cols = ["Year", "Month", "Week", "Day", "Dayofyear", "Is_month_end", "Is_month_start",
             "Is_quarter_end", "Is_quarter_start", "Is_year_end", "Is_year_start", "Elapsed"]
raw_time = raw_time.drop(columns=date_cols, level=0)

In [ ]:
raw_time.isnull().sum().sort_values(ascending=False)/len(raw)

In [ ]:
columns = raw_time.columns.to_list()
print(columns)

In [ ]:
raw_time = raw_time.copy(True)

In [ ]:
raw_time.loc[:, (slice(None),'^NDX')]

## NQ futures (overnight)

The Nasdaq keeps trading at night with the NQ futures (about 23h a day). We use 1-minute bars from Kaggle ("NQ Futures - 1min Bar 2022 2025", TGT Analytics, CC0), loaded at the top with `kagglehub` (`NQ`), so we don't need to download the CSV by hand. The timestamps are already in New York time, so we parse them as they are (no time-zone conversion).

**Anti-leakage rule:** one fixed cutoff, 09:00 ET of day `t`. We never use an NQ bar after 09:00 of day `t`, because during the NY session NQ and `^NDX` move almost the same. We don't use `Vwap_RTH` and `Vwap_ETH`.

In [ ]:
nq = NQ[["timestamp ET", "open", "high", "low", "close", "volume"]]
nq = nq.rename(columns={"timestamp ET": "timestamp"})
nq["timestamp"] = pd.to_datetime(nq["timestamp"], format="%m/%d/%Y %H:%M")
nq = nq.sort_values("timestamp").reset_index(drop=True)

In [ ]:
print(nq["timestamp"].min(), "->", nq["timestamp"].max(), len(nq))
print(nq.isnull().sum())

In [ ]:
# bars per calendar day (a full day is about 1380 bars)
bars_per_day = nq.groupby(nq["timestamp"].dt.normalize()).size()
print(bars_per_day.describe())

We observe that the total days is 923, instead of 1080, which is the supposed total length. We explain this beacuse of the saturdays were the market closes. Also, we observe that the mean is 1136, instead of 1380, however this is explain by the days that close early. Which is described by the percentiles.

In [ ]:
# bars per hour of the day: the night hours are there, and 17:00-18:00 is the daily break
print(nq.groupby(nq["timestamp"].dt.hour).size())

In [ ]:
# the last day of the file is cut
print(bars_per_day.tail(3))

### Timestamp: start or end of the bar?

Globex opens at 18:00 and closes at 17:00. If the timestamp is the **end** of the minute, the first bar of a session is 18:01 and the last one is 17:00. If it is the **start**, the first bar is 18:00 and the last one is 16:59. So we count the bars stamped 17:00 and 18:00.

In [ ]:
print(nq.head(3))
hhmm = nq["timestamp"].dt.strftime("%H:%M")
print(hhmm[hhmm.isin(["16:59", "17:00", "18:00", "18:01"])].value_counts())

In [ ]:
stamp_is_end = (hhmm == "17:00").sum() > (hhmm == "18:00").sum()
print("timestamp is the END of the bar:", stamp_is_end)

Result: the timestamp is the bar **end**. There are bars at 18:01 and 17:00 and no bar at 18:00. So the last bar we can use is the one stamped 09:00 (`timestamp <= 09:00`).

To avoid mistakes we build `bar_start` and `bar_end` for each bar and we always work with those two.

In [ ]:
one_min = pd.Timedelta(minutes=1)
nq["bar_end"] = nq["timestamp"] if stamp_is_end else nq["timestamp"] + one_min
nq["bar_start"] = nq["bar_end"] - one_min

### Sessions

We add 6 hours to the time, so 18:00 of the evening before becomes 00:00 of day `t`. In this shifted time the overnight window of day `t` is 00:00-15:00 (18:00 to 09:00 in NY), Asia is 00:00-09:00 and Europe is 09:00-15:00. Sunday evening goes to Monday.

In [ ]:
shifted = nq["bar_start"] + pd.Timedelta(hours=6)
nq["session_date"] = shifted.dt.normalize()
nq["h"] = shifted.dt.hour

In [ ]:
# overnight window, without the last (cut) day
last_day = nq["session_date"].max()
ovn = nq[(nq["h"] < 15) & (nq["session_date"] < last_day)].copy()
asia = ovn[ovn["h"] < 9]
europe = ovn[ovn["h"] >= 9]
print("last day dropped:", last_day.date())

In [ ]:
# anti-leakage: the latest bar used for day t ends at 09:00 of day t or before
last_used = ovn.groupby("session_date")["bar_end"].max()
assert (last_used <= last_used.index + pd.Timedelta(hours=9)).all()
print(last_used.dt.strftime("%H:%M").value_counts().head())

In [ ]:
agg = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
o = ovn.groupby("session_date").agg(agg)
a = asia.groupby("session_date").agg(agg)
e = europe.groupby("session_date").agg(agg)

We keep only the days where `^NDX` trades (the futures also trade on some US holidays) and we drop the days with clearly too few overnight bars (a full night is 900 bars).

In [ ]:
n_bars = ovn.groupby("session_date").size()
few = n_bars[n_bars < 450].index
keep = o.index[o.index.isin(df_nasdaq.index) & ~o.index.isin(few)]
print("NQ sessions:", len(o), "| not a ^NDX day:", (~o.index.isin(df_nasdaq.index)).sum(),
      "| too few bars:", len(few), "| kept:", len(keep))

In [ ]:
o = o.loc[keep]
a = a.reindex(keep)
e = e.reindex(keep)
# p09: NQ close of the last bar before the 09:00 cutoff
p09 = o["close"]

### Overnight features

All the features are returns, distances in % or ratios, never price levels, so they mean the same in 2023 and in 2025. We always compare NQ with NQ.

Returns of each session:

In [ ]:
nq_feat = pd.DataFrame(index=keep)
nq_feat["asia_ret"] = np.log(a["close"] / a["open"])
nq_feat["europe_ret"] = np.log(e["close"] / e["open"])
nq_feat["ovn_ret"] = np.log(p09 / o["open"])

Distance from the 09:00 price to the high and low of each session (support / resistance), and where the price is in the range of the night (0 = low, 1 = high).

In [ ]:
nq_feat["asia_high_dist"] = np.log(a["high"] / p09)
nq_feat["asia_low_dist"] = np.log(a["low"] / p09)
nq_feat["europe_high_dist"] = np.log(e["high"] / p09)
nq_feat["europe_low_dist"] = np.log(e["low"] / p09)
nq_feat["ovn_high_dist"] = np.log(o["high"] / p09)
nq_feat["ovn_low_dist"] = np.log(o["low"] / p09)
nq_feat["ovn_range_pos"] = (p09 - o["low"]) / (o["high"] - o["low"])

High and low of NQ during the NY session (09:30-16:00) of the previous trading day. This session is fully finished before the cutoff.

In [ ]:
minute = nq["bar_start"].dt.hour * 60 + nq["bar_start"].dt.minute
ny = nq[(minute >= 9 * 60 + 30) & (minute < 16 * 60)]
ny = ny.groupby(ny["bar_start"].dt.normalize()).agg({"high": "max", "low": "min", "close": "last"})
# previous trading day of ^NDX
prev_ny = ny.reindex(df_nasdaq.index).shift(1).reindex(keep)

In [ ]:
nq_feat["prev_ny_high_dist"] = np.log(prev_ny["high"] / p09)
nq_feat["prev_ny_low_dist"] = np.log(prev_ny["low"] / p09)

Volume. The "previous 20 days" means use `.shift(1).rolling(20)`, so day `t` is not in its own mean.

In [ ]:
ovn_volume = o["volume"]
nq_feat["ovn_volume_rel"] = ovn_volume / ovn_volume.shift(1).rolling(20).mean()
nq_feat["asia_volume_share"] = a["volume"] / ovn_volume
nq_feat["europe_volume_share"] = e["volume"] / ovn_volume

In [ ]:
# volume of the last hour before the cutoff (08:00-09:00), reaction to the 08:30 macro news
last_hour = ovn[ovn["h"] == 14].groupby("session_date")["volume"].sum().reindex(keep)
nq_feat["last_hour_volume_rel"] = last_hour / last_hour.shift(1).rolling(20).mean()

Volatility: std of the 1-minute log returns of the night, and the size of the night's range.

In [ ]:
ovn["lr"] = np.log(ovn["close"]).groupby(ovn["session_date"]).diff()
nq_feat["ovn_vol"] = ovn.groupby("session_date")["lr"].std().reindex(keep)
nq_feat["ovn_range"] = np.log(o["high"] / o["low"])

### Roll days

The contract changes every quarter (March, June, September, December). If the file is not adjusted, the price jumps between two sessions on the roll day. The overnight window is always inside one contract, so only `prev_ny_*_dist` would be affected (yesterday's contract against today's price).

First we look at the gap between the NQ close at 16:00 of the previous day and the first overnight open.

In [ ]:
roll_gap = np.log(o["open"] / prev_ny["close"])
print(roll_gap.abs().sort_values(ascending=False).head(20))

The biggest gaps are news days (the tariff days of April 2025, earnings after the close...), not the quarterly roll dates. So this gap alone can't find the rolls.

A cleaner check: the difference in points between NQ and `^NDX` at the 16:00 close. A roll moves it by 150-250 points from one day to the next. We only use `^NDX` here for this check, not for a feature.

In [ ]:
spread = (ny["close"] - df_nasdaq[("Close", "^NDX")]).dropna()
print(spread.resample("QS").mean().round(0))
print(spread.diff().abs().sort_values(ascending=False).head(5))

In [ ]:
ROLL_JUMP = 100
roll_days = spread.index[spread.diff().abs() > ROLL_JUMP]
print(len(roll_days), "roll days:", list(roll_days.date))

Result: there is no roll jump in this file. The difference goes down slowly (from about 2,700 points at the start of 2023 to about 300 at the end of 2025) and the biggest change in one day is under 40 points, so the series is already a back-adjusted continuous contract. No roll day to drop.

Limit to keep in mind: because of this adjustment the NQ prices of the first months are too high (about 25%), so the overnight returns of 2023 are a bit smaller than the real ones (about 20% smaller at the start, almost nothing at the end).

In [ ]:
nq_feat.describe().T

## Target

We decide at 09:00 ET of day `t`, we enter at the NY open of day `t` (09:30) and we exit at the NY close of the same day (16:00).

The target is the open-to-close log return of `^NDX`, and the label is 1 if it is positive, 0 if not. It is open-to-close on purpose: the overnight move is already in the open price. The `Open` and `Close` of `^NDX` of day `t` are only used here, never as features.

In [ ]:
ndx = raw_drop.set_index('Date')
close = ndx['Close']['^NDX']
open_ = ndx['Open']['^NDX']

In [ ]:
open_

In [ ]:
ret_ny = np.log(close / open_)

In [ ]:
ret_ny

In [ ]:
raw_time = raw_time.set_index("Date")

In [ ]:
raw_time

In [ ]:
data = raw_time.copy()
# both have the Date index, so pandas aligns them by date
data["ret_ny"] = ret_ny

In [ ]:
r = data["ret_ny"]
data["y_ny"] = (r > 0).astype(float).where(r.notna())

In [ ]:
print(data[["ret_ny", "y_ny"]].tail(7))

In [ ]:
print(data.columns.tolist())
print(data.index.min(), data.index.max(), len(data))

The target doesn't look ahead anymore, so the last rows keep their target. We only remove the rows without target (if any).

In [ ]:
target_cols = [("ret_ny", ""), ("y_ny", "")]

data = data.dropna(subset=[("y_ny", "")])

## Daily features

Context from the day before. All the daily US data closes at 16:00 or later, so everything here is shifted by 1 day (`shift(1)`): for day `t` we only use values up to the close of `t-1`.

Nasdaq-100: momentum over 1, 5 and 20 days, the NY session return of the day before, and the 20-day volatility of the daily returns.

In [ ]:
ndx_close = data[("Close", "^NDX")]
for n in (1, 5, 20):
    data[(f"ndx_ret_{n}d", "")] = np.log(ndx_close / ndx_close.shift(n)).shift(1)

In [ ]:
data[("ndx_ret_ny_prev", "")] = data[("ret_ny", "")].shift(1)
data[("ndx_vol_20d", "")] = np.log(ndx_close).diff().rolling(20).std().shift(1)

Macro: level for the volatility indices and the yields, daily log return for the dollar, oil and gold. Some of them don't trade on a few Nasdaq days (bond holidays), so we forward fill first (it only uses the past). Silver and the 5y / 30y yields are not used.

In [ ]:
macro_close = data["Close"][["^VIX", "^VXN", "^TNX", "^IRX", "DX-Y.NYB", "CL=F", "GC=F"]].ffill()

In [ ]:
data[("vix", "")] = macro_close["^VIX"].shift(1)
data[("vxn", "")] = macro_close["^VXN"].shift(1)
data[("yield_10y", "")] = macro_close["^TNX"].shift(1)
data[("yield_3m", "")] = macro_close["^IRX"].shift(1)

In [ ]:
data[("dollar_ret", "")] = np.log(macro_close["DX-Y.NYB"]).diff().shift(1)
data[("oil_ret", "")] = np.log(macro_close["CL=F"]).diff().shift(1)
data[("gold_ret", "")] = np.log(macro_close["GC=F"]).diff().shift(1)

Bitcoin (risk sentiment). A yfinance `BTC-USD` daily bar dated `D` covers 00:00-24:00 UTC, so it closes around 19:00-20:00 ET of day `D`: the bar dated `t-1` is known before the 09:00 cutoff of day `t`, the bar dated `t` is not.

`btc_ret` is the BTC return since the last trading day (Monday's value covers the whole weekend). `btc_vol_7d` is the std of the BTC daily log returns over the 7 calendar days ending at `t-1`.

In [ ]:
days = pd.Series(data.index, index=data.index)
btc_last = days - pd.Timedelta(days=1)             # calendar day t-1
btc_first = days.shift(1) - pd.Timedelta(days=1)   # calendar day before the previous trading day
# the BTC values of day t only use bars dated t-1 or before
assert (btc_last < days).all() and (btc_first.dropna() < btc_last[btc_first.notna()]).all()

In [ ]:
data[("btc_ret", "")] = np.log(btc_close.reindex(btc_last).values / btc_close.reindex(btc_first).values)
btc_vol = np.log(btc_close).diff().rolling(7).std()
data[("btc_vol_7d", "")] = btc_vol.reindex(btc_last).values

Stocks: daily log return of each stock, shifted by 1. We keep only the stocks with no missing values in the period. They are kept for a PCA that we will do later, fitted on train only.

In [ ]:
not_stocks = tickers1 + ["^NDX"]
stocks = [t for t in data["Close"].columns if t not in not_stocks]
stock_close = data["Close"][stocks].dropna(axis=1)
print(len(stocks), "stocks,", stock_close.shape[1], "with no missing values")

In [ ]:
stock_ret = np.log(stock_close).diff().shift(1)
stock_ret.columns = pd.MultiIndex.from_product([["stock_ret"], stock_ret.columns])

Breadth: % of those stocks with a positive return the day before.

In [ ]:
data[("pct_up", "")] = ((stock_ret > 0).mean(axis=1) * 100).where(stock_ret.notna().all(axis=1))

## Join

The Open, Close and Volume levels are not features (they are replaced by the returns above), so we drop them. Then we do an inner join of the daily table and the NQ features on the `^NDX` trading dates.

In [ ]:
data = data.drop(columns=["Open", "Close", "Volume"], level=0)
data = pd.concat([data, stock_ret], axis=1)

In [ ]:
nq_feat.columns = pd.MultiIndex.from_product([nq_feat.columns, [""]])
nq_cols = nq_feat.columns.tolist()
print("daily rows:", len(data), "| NQ rows:", len(nq_feat))
data = data.join(nq_feat, how="inner")
print("after the join:", len(data))

In [ ]:
# drop the roll days
n_roll = data.index.isin(roll_days).sum()
data = data[~data.index.isin(roll_days)]
print("roll days dropped:", n_roll)

The first rows have no 20-day means (warm-up of the NQ file), so they have missing values. We remove the rows with missing values.

In [ ]:
nulls = data.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False))

In [ ]:
n_before = len(data)
data = data.dropna()
print("rows dropped:", n_before - len(data), "| first day:", data.index.min().date())

In [ ]:
# anti-leakage: no price level of day t in the features
feature_cols = [c for c in data.columns if c not in target_cols]
assert ("Close", "^NDX") not in feature_cols and ("Open", "^NDX") not in feature_cols
assert not data.columns.get_level_values(0).isin(["Open", "Close", "Volume"]).any()

In [ ]:
print(data.shape)
print(data.index.min(), data.index.max())

In [ ]:
data.to_feather("../data/final_data_ny.feather")

## Train / test split : moved to DATA_SPLITTING.ipynb

The split is chronological: the test set is the last 20% of the rows and we don't touch it until the end. The test set starts on 2025-05-16 (144 rows, train has 578).

The target doesn't look ahead anymore (it ends at the close of day `t`), so we don't need to remove rows of train. We keep `gap = 1` in `TimeSeriesSplit`.

Class balance: in train the NY session goes up 55.5% of the days (56.9% in test). So a model that always says "up" already gets that accuracy, that's the baseline to beat.

**Known limitation:** the table has one row per day and the NQ file only covers Dec 2022 - Dec 2025, so we only have 722 rows (578 for train and 144 for test). It is small. Possible options for later: a longer NQ source (since 2008, not on Kaggle), or a panel with one row per stock per day.